# Adaptive AI Agents for Fraud Detection (Workshop Notebook)

This notebook is a **starter template** for:
- Exploratory data analysis (EDA)
- Building a baseline fraud detector
- Adding **message/text-derived** features (keyword / ML / LLM-assisted)
- Evaluating with an explicit **cost function**

Data files:
- `../data/train/*.csv`
- `../data/test/*.csv`

Recommended team split:
- Business: define the cost function + operating point (review budget)
- Tech: build features + model + evaluation
- Both: explanations + final story/demo


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

DATA_DIR = Path("..") / "data"
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

TRAIN_DIR, TEST_DIR

In [ ]:
def weak_phish_heuristic(sender: str, text: str) -> int:
    s = (sender or "").lower()
    t = (text or "").lower()
    score = 0
    if "http" in t or "https" in t:
        score += 1
    for k in ["verify", "urgent", "locked", "confirm", "fee", "pay", "password", "re-authenticate", "voucher"]:
        if k in t:
            score += 1
    if any(x in s for x in ["security", "support", "delivery", "verify"]):
        score += 1
    return int(score >= 2)


def load_split(
    split_dir: Path,
    transactions_name: str = "transactions.csv",
    messages_name: str = "messages.csv",
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    users = pd.read_csv(split_dir / "users.csv")
    transactions = pd.read_csv(split_dir / transactions_name)
    messages = pd.read_csv(split_dir / messages_name)

    transactions["timestamp"] = pd.to_datetime(transactions["timestamp"], utc=True)
    messages["timestamp"] = pd.to_datetime(messages["timestamp"], utc=True)

    # If public test messages are used (no label), create a weak baseline proxy.
    if "is_phishing" not in messages.columns:
        messages["is_phishing"] = messages.apply(lambda r: weak_phish_heuristic(r.get("sender", ""), r.get("message_text", "")), axis=1)

    return users, transactions, messages


train_users, train_txns, train_msgs = load_split(TRAIN_DIR)

# Prefer public test files (no labels) if present
test_txn_name = "transactions_public.csv" if (TEST_DIR / "transactions_public.csv").exists() else "transactions.csv"
test_msg_name = "messages_public.csv" if (TEST_DIR / "messages_public.csv").exists() else "messages.csv"
test_users, test_txns, test_msgs = load_split(TEST_DIR, transactions_name=test_txn_name, messages_name=test_msg_name)

test_txn_labels = pd.read_csv(TEST_DIR / "transactions_labels.csv") if (TEST_DIR / "transactions_labels.csv").exists() else None

print("train:", train_users.shape, train_txns.shape, train_msgs.shape)
print("test :", test_users.shape, test_txns.shape, test_msgs.shape)
print("test labels:", None if test_txn_labels is None else test_txn_labels.shape)

train_txns.head(3)

## Quick sanity checks

In [ ]:
display(train_txns["is_fraud"].value_counts(dropna=False))
display(train_msgs["is_phishing"].value_counts(dropna=False))

print("fraud rate:", train_txns["is_fraud"].mean().round(4))
print("phish msg rate:", train_msgs["is_phishing"].mean().round(4))

missing = train_txns.isna().mean().sort_values(ascending=False).head(12)
missing

## EDA (transactions)

Look for patterns that are plausible signals:
- Amount outliers
- Online entry mode
- New device / new merchant
- International vs domestic
- Transfers to new beneficiaries
- Odd hours + spikes (velocity)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

sns.histplot(data=train_txns[train_txns["direction"] == "debit"], x="amount_eur", hue="is_fraud", bins=60, ax=axes[0, 0], log_scale=(False, True))
axes[0, 0].set_title("Debit amount distribution (log y)")

sns.countplot(data=train_txns, x="entry_mode", hue="is_fraud", ax=axes[0, 1])
axes[0, 1].set_title("Entry mode")
axes[0, 1].tick_params(axis="x", rotation=30)

sns.countplot(data=train_txns, x="channel", hue="is_fraud", ax=axes[1, 0])
axes[1, 0].set_title("Channel")
axes[1, 0].tick_params(axis="x", rotation=20)

sns.countplot(data=train_txns, x="is_international", hue="is_fraud", ax=axes[1, 1])
axes[1, 1].set_title("International")

plt.tight_layout()
plt.show()

train_txns.groupby(["merchant_category"])['is_fraud'].mean().sort_values(ascending=False).head(12)

## EDA (messages)

A common pattern in real systems: fraud attempts correlate with **phishing / social-engineering messages**.

Try:
- Sender analysis
- Keyword analysis
- "Phishing in last X hours" before a transaction


In [ ]:
train_msgs.groupby(["sender"])['is_phishing'].mean().sort_values(ascending=False).head(12)

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
sns.countplot(data=train_msgs, x="channel", hue="is_phishing", ax=ax)
ax.set_title("Messages by channel")
plt.show()

train_msgs.sample(5, random_state=0)[["sender", "channel", "message_text", "is_phishing"]]

## Feature engineering

Below is a simple way to turn message streams into transaction-level features:
- count of messages in last 24h / 7d
- count of phishing messages in last 24h / 7d
- simple keyword score (teams can replace with TF‑IDF or LLM extraction)


In [ ]:
SUSPICIOUS_KEYWORDS = [
    "verify",
    "urgent",
    "locked",
    "confirm",
    "unusual",
    "blocked",
    "voucher",
    "fee",
    "password",
    "code",
    "account",
    "security",
]


def keyword_score(text: str) -> int:
    if not isinstance(text, str) or not text:
        return 0
    t = text.lower()
    return sum(1 for k in SUSPICIOUS_KEYWORDS if k in t)


def build_message_features(transactions: pd.DataFrame, messages: pd.DataFrame) -> pd.DataFrame:
    tx = transactions[["transaction_id", "user_id", "timestamp"]].copy()
    msg = messages[["user_id", "timestamp", "is_phishing", "message_text"]].copy()
    msg["kw_score"] = msg["message_text"].map(keyword_score)

    # Work user-by-user to compute windowed counts using cumulative sums.
    tx = tx.sort_values(["user_id", "timestamp"]).reset_index(drop=True)
    msg = msg.sort_values(["user_id", "timestamp"]).reset_index(drop=True)

    features = []
    for user_id, tx_u in tx.groupby("user_id", sort=False):
        msg_u = msg[msg["user_id"] == user_id]
        t_tx = tx_u["timestamp"].to_numpy(dtype="datetime64[ns]")
        t_msg = msg_u["timestamp"].to_numpy(dtype="datetime64[ns]")

        # Handle users with no messages
        if len(t_msg) == 0:
            out = pd.DataFrame(
                {
                    "transaction_id": tx_u["transaction_id"].to_numpy(),
                    "msg_count_24h": 0,
                    "msg_count_7d": 0,
                    "phish_count_24h": 0,
                    "phish_count_7d": 0,
                    "kw_score_24h": 0,
                    "kw_score_7d": 0,
                }
            )
            features.append(out)
            continue

        is_phish = msg_u["is_phishing"].to_numpy(dtype=int)
        kw = msg_u["kw_score"].to_numpy(dtype=int)

        cum_msg = np.arange(1, len(t_msg) + 1)
        cum_phish = np.cumsum(is_phish)
        cum_kw = np.cumsum(kw)

        def window_counts(window: np.timedelta64):
            right = np.searchsorted(t_msg, t_tx, side="right")
            left = np.searchsorted(t_msg, t_tx - window, side="right")

            msg_count = right - left
            phish_count = (cum_phish[np.maximum(right - 1, 0)] - np.where(left > 0, cum_phish[left - 1], 0))
            kw_score_sum = (cum_kw[np.maximum(right - 1, 0)] - np.where(left > 0, cum_kw[left - 1], 0))

            # Fix rows where right == 0 (no messages before txn): the above indexing used element 0.
            zero_mask = right == 0
            if np.any(zero_mask):
                phish_count = phish_count.astype(int)
                kw_score_sum = kw_score_sum.astype(int)
                phish_count[zero_mask] = 0
                kw_score_sum[zero_mask] = 0

            return msg_count.astype(int), phish_count.astype(int), kw_score_sum.astype(int)

        msg24, phish24, kw24 = window_counts(np.timedelta64(24, "h"))
        msg7d, phish7d, kw7d = window_counts(np.timedelta64(7, "D"))

        out = pd.DataFrame(
            {
                "transaction_id": tx_u["transaction_id"].to_numpy(),
                "msg_count_24h": msg24,
                "msg_count_7d": msg7d,
                "phish_count_24h": phish24,
                "phish_count_7d": phish7d,
                "kw_score_24h": kw24,
                "kw_score_7d": kw7d,
            }
        )
        features.append(out)

    return pd.concat(features, ignore_index=True)


train_msg_feat = build_message_features(train_txns, train_msgs)
test_msg_feat = build_message_features(test_txns, test_msgs)

train_feat = train_txns.merge(train_msg_feat, on="transaction_id", how="left")
test_feat = test_txns.merge(test_msg_feat, on="transaction_id", how="left")

train_feat[["transaction_id", "msg_count_24h", "phish_count_24h", "kw_score_24h"]].head(5)

## Model training (baseline)

Below: a simple `LogisticRegression` baseline with one-hot encoded categorical features.

Teams can swap in:
- Tree models (RandomForest, GradientBoosting)
- Better text models (TF‑IDF on messages, LLM classification)
- Cost-sensitive learning / calibrated scoring


In [ ]:
@dataclass(frozen=True)
class TrainConfig:
    label_col: str = "is_fraud"
    time_col: str = "timestamp"


cfg = TrainConfig()

# Time-based validation split (more realistic than random split)
cutoff = train_feat[cfg.time_col].quantile(0.85)
train_part = train_feat[train_feat[cfg.time_col] <= cutoff].copy()
valid_part = train_feat[train_feat[cfg.time_col] > cutoff].copy()

print("cutoff:", cutoff)
print("train/valid sizes:", train_part.shape, valid_part.shape)

y_train = train_part[cfg.label_col].astype(int)
y_valid = valid_part[cfg.label_col].astype(int)

drop_cols = [
    cfg.label_col,
    "fraud_type",
    "fraud_loss_eur",
    "transaction_id",
    "timestamp",
]

X_train = train_part.drop(columns=[c for c in drop_cols if c in train_part.columns])
X_valid = valid_part.drop(columns=[c for c in drop_cols if c in valid_part.columns])
X_test = test_feat.drop(columns=[c for c in drop_cols if c in test_feat.columns])

categorical_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("scale", StandardScaler(with_mean=False)),
            ]),
            numeric_cols,
        ),
        (
            "cat",
            Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_cols,
        ),
    ],
    remainder="drop",
)

model = LogisticRegression(max_iter=2000, class_weight="balanced")

clf = Pipeline(
    steps=[
        ("prep", preprocess),
        ("model", model),
    ]
)

clf.fit(X_train, y_train)

p_valid = clf.predict_proba(X_valid)[:, 1]
print("ROC-AUC:", roc_auc_score(y_valid, p_valid).round(4))
print("PR-AUC :", average_precision_score(y_valid, p_valid).round(4))

In [ ]:
def pick_threshold_cost(valid_df: pd.DataFrame, p: np.ndarray, y_true: np.ndarray) -> tuple[float, pd.DataFrame]:
    """Pick a threshold that minimizes: review_cost * alerts + loss * missed_fraud."""
    thresholds = np.linspace(0.01, 0.99, 99)
    rows = []
    for th in thresholds:
        y_hat = (p >= th).astype(int)
        alerts = int(y_hat.sum())
        missed = int(((y_true == 1) & (y_hat == 0)).sum())

        # Use the dataset-provided proxies
        review_cost = float((valid_df["review_cost_eur"] * y_hat).sum())
        missed_loss = float((valid_df.loc[(y_true == 1) & (y_hat == 0), "potential_loss_eur"]).sum())
        total_cost = review_cost + missed_loss

        precision = 0.0 if alerts == 0 else float(((y_true == 1) & (y_hat == 1)).sum()) / alerts
        recall = 0.0 if y_true.sum() == 0 else float(((y_true == 1) & (y_hat == 1)).sum()) / float(y_true.sum())
        rows.append({"threshold": th, "alerts": alerts, "missed": missed, "precision": precision, "recall": recall, "total_cost": total_cost})

    curve = pd.DataFrame(rows).sort_values("threshold")
    best = curve.sort_values("total_cost").iloc[0]
    return float(best["threshold"]), curve


best_th, cost_curve = pick_threshold_cost(valid_part, p_valid, y_valid.to_numpy())
best_th, cost_curve.sort_values("total_cost").head(5)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.plot(cost_curve["threshold"], cost_curve["total_cost"], label="total_cost")
ax.axvline(best_th, color="red", linestyle="--", label=f"best={best_th:.2f}")
ax.set_title("Cost vs threshold")
ax.set_xlabel("threshold")
ax.set_ylabel("cost (EUR proxy)")
ax.legend()
plt.show()

y_hat_valid = (p_valid >= best_th).astype(int)
print(classification_report(y_valid, y_hat_valid, digits=3))
confusion_matrix(y_valid, y_hat_valid)

## Apply to test + simple explanations

In a real bank workflow, the model output is only part of the story:
- you need **explanations** for analysts and customers
- you need a decision policy (review vs. block vs. step-up auth)
- you need monitoring for drift


In [ ]:
p_test = clf.predict_proba(X_test)[:, 1]
test_scored = test_feat.copy()
test_scored["risk_score"] = p_test
test_scored["flagged"] = (test_scored["risk_score"] >= best_th).astype(int)


def explain_alert(tx_row: pd.Series) -> str:
    reasons = []
    if tx_row.get("entry_mode") == "online":
        reasons.append("online card entry")
    if int(tx_row.get("device_seen_before", 1)) == 0:
        reasons.append("new / unseen device")
    if int(tx_row.get("merchant_seen_before", 1)) == 0:
        reasons.append("new merchant")
    if int(tx_row.get("is_international", 0)) == 1:
        reasons.append("international merchant")
    if float(tx_row.get("amount_eur", 0.0)) > 600:
        reasons.append("high amount")
    if int(tx_row.get("phish_count_24h", 0)) > 0:
        reasons.append("recent phishing messages")
    if int(tx_row.get("kw_score_24h", 0)) >= 3:
        reasons.append("suspicious message keywords")

    if not reasons:
        reasons = ["anomalous pattern vs. history"]

    return "; ".join(reasons)


alerts = test_scored[test_scored["flagged"] == 1].copy()
alerts = alerts.sort_values("risk_score", ascending=False).head(15)
alerts["explanation"] = alerts.apply(explain_alert, axis=1)

# If we loaded public test transactions, merge labels (facilitator-only) for evaluation.
if "is_fraud" not in alerts.columns and test_txn_labels is not None:
    alerts = alerts.merge(test_txn_labels[["transaction_id", "is_fraud"]], on="transaction_id", how="left")

cols = ["timestamp", "user_id", "amount_eur", "merchant_name", "merchant_category", "entry_mode", "ip_country", "risk_score", "explanation"]
if "is_fraud" in alerts.columns:
    cols.append("is_fraud")

alerts[cols].head(15)

## Optional: LLM-assisted text understanding (plug-in)

Teams can use an LLM to extract higher-quality signals from `messages.message_text`, for example:
- classify intent: OTP / delivery scam / IT reset / invoice manipulation
- extract entities: merchant names, amounts, urgency cues
- generate rules/features: e.g. a list of risk keywords by language
- draft analyst-friendly explanations

Example prompt idea (provider-agnostic):

```
You are helping a bank fraud team.
Task: label the message as PHISHING or BENIGN and extract signals.
Return JSON: {"is_phishing":0|1, "intent":"...", "signals":[...]}.
Message: <MESSAGE_TEXT>
```

Then aggregate those signals into windowed features ("phishing in last 24h"), similar to the code above.


## Agentic approach (LLM-powered Alert Agent)

A practical way to include an LLM **without** replacing your entire model:

- Use a normal classifier/rule-engine for the **risk score**
- Use an LLM “Alert Agent” to produce structured outputs:
  - explanation
  - recommended action (`review` / `step_up_auth` / `block`)
  - rationale bullets

This repo ships an offline `MockLLM` so the agent pipeline runs without API keys.
To plug in a real endpoint, set env vars (see `agents/README.md`).


In [ ]:
import sys
from pathlib import Path

# If your Jupyter cwd is `notebooks/`, add repo root so `import agents` works.
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from agents.llm import llm_from_env
from agents.alert_agent import AlertAgent

# Default is offline mock. For real usage, set env vars before starting Jupyter.
llm = llm_from_env()
agent = AlertAgent(llm=llm, messages=test_msgs, message_window_hours=24)

sample_alerts = alerts.head(10).copy()
agent_rows = []
for _, r in sample_alerts.iterrows():
    d = agent.decide(r)
    agent_rows.append(
        {
            "transaction_id": r["transaction_id"],
            "risk_score": float(r["risk_score"]),
            "agent_severity": d.severity,
            "agent_action": d.recommended_action,
            "agent_explanation": d.explanation,
            "agent_rationale": "; ".join(d.rationale_bullets),
        }
    )

pd.DataFrame(agent_rows)